In [1]:
import requests
from bs4 import BeautifulSoup
import os
import time
from datetime import datetime

In [ ]:
URL = "https://en.wikipedia.org/wiki/Lionel_Messi"

headers = {
    "User-Agent": "Aula-CPA"
}

#prefixos indesejados para links internos
INVALID_PREFIXES = (
    "/wiki/Main_Page",
    "/wiki/Wikipedia",
    "/wiki/Help",
    "/wiki/Portal",
    "/wiki/Special",
    "/wiki/Talk",
    "/wiki/Category",
    "/wiki/File",
    "/wiki/Template",
)

#Cria as pastas
output_dir = "data"
html_dir = os.path.join(output_dir, "html")

os.makedirs(output_dir, exist_ok=True)

#Criar dicionário para armazenar os metadados
metadata = {}

#limpar html antigo
import shutil
if os.path.exists(html_dir):
    shutil.rmtree(html_dir)

os.makedirs(html_dir)

#baixar HTML principal
response = requests.get(URL, headers=headers)
response_soup = BeautifulSoup(response.text, "html.parser")

main_html_path = os.path.join(html_dir, "main.html")

with open(main_html_path, "w", encoding="utf-8") as f:
    f.write(response_soup.prettify())

metadata["main.html"] = {
    "url": URL,
    "timestamp": datetime.now()
}

print("HTML principal salvo")

#extrair links A PARTIR do HTML salvo
with open(main_html_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]

    if (
        href.startswith("/wiki/")
        and ":" not in href
        and "#" not in href
        and not href.startswith(INVALID_PREFIXES)  # reutiliza a constante já definida
    ):
        links.add("https://en.wikipedia.org" + href)

links = list(links)

print(f"{len(links)} links encontrados")

# Criar uma sessão para reutilizar conexões
session = requests.Session()

#baixar páginas dos links
for i, link in enumerate(links[:1000]):  # limitar a 1000 links para facilitar a vida de todos os envolvidos
                                         # caso queira baixar tudo, basta remover[:1000], mas vai demorar o dobro
    try:
        r = session.get(link, headers=headers)

        file_name = f"page_{i}.html"
        file_path = os.path.join(html_dir, file_name)  # reutiliza file_name.

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r.text)

        metadata[file_name] = {
            "url": link,
            "timestamp": datetime.now()
        }

        time.sleep(0.3)

    except Exception:
        continue

print("Download concluído")

HTML principal salvo
1957 links encontrados
Download concluído


In [3]:
import pandas as pd

#Garante o caminho
html_dir = os.path.join(output_dir, "html")

#Sempre reinicia (evita duplicação)
data_pages = []
data_images = []

for file in sorted(os.listdir(html_dir), key=lambda f: (0 if f == "main.html" else 1, f)):
    path = os.path.join(html_dir, file)

    if not os.path.isfile(path):
        continue

    with open(path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    #título
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    #conteúdo principal
    content = soup.find("div", {"id": "mw-content-text"})
    first_paragraph = None

    if content:
        paragraphs = content.find_all("p", recursive=True)

        for p in paragraphs:
            text = p.get_text(strip=True)
            if text:
                first_paragraph = text
                break

    #imagens
    images = [
        "https:" + img["src"]
        for img in soup.find_all("img")
        if img.get("src") and img["src"].startswith("//upload")
    ]

    link_principal = metadata[file]["url"]
    timestamp = metadata[file]["timestamp"]

    #links internos
    links_tags = soup.find_all("a", href=True)

    internal_links = [
        "https://en.wikipedia.org" + a["href"]
        for a in links_tags
        if (
            a["href"].startswith("/wiki/")
            and ":" not in a["href"]
            and "#" not in a["href"]
            and "https://en.wikipedia.org" + a["href"] != link_principal
            and not a["href"].startswith(INVALID_PREFIXES)
        )
    ]

    #salva uma vez só
    data_pages.append({
        "title": title,
        "first_paragraph": first_paragraph,
        "link_principal": link_principal,
        "internal_links": " | ".join(
            [f"[{link}]" for link in internal_links]
        ),
        "timestamp": timestamp
    })

    #imagens
    for img in images:
        data_images.append({
            "image_url": img
        })

In [4]:
#imprime com quebra de linha os links
for link in internal_links:
    print(link)

https://en.wikipedia.org/wiki/Spanish_name
https://en.wikipedia.org/wiki/Surname
https://en.wikipedia.org/wiki/San_Luis_Potos%C3%AD_(San_Luis_Potos%C3%AD)
https://en.wikipedia.org/wiki/Forward_(association_football)
https://en.wikipedia.org/wiki/Danubio_F.C.
https://en.wikipedia.org/wiki/Olympiacos_F.C.
https://en.wikipedia.org/wiki/FC_Shakhtar_Donetsk
https://en.wikipedia.org/wiki/Manchester_City_F.C.
https://en.wikipedia.org/wiki/FC_Dnipro
https://en.wikipedia.org/wiki/Chicago_Fire_FC
https://en.wikipedia.org/wiki/Aris_Thessaloniki_F.C.
https://en.wikipedia.org/wiki/Aris_Thessaloniki_F.C.
https://en.wikipedia.org/wiki/C.F._Pachuca
https://en.wikipedia.org/wiki/Club_Le%C3%B3n
https://en.wikipedia.org/wiki/Rayo_Vallecano
https://en.wikipedia.org/wiki/Mexico_national_football_team
https://en.wikipedia.org/wiki/CONCACAF_Gold_Cup
https://en.wikipedia.org/wiki/2007_CONCACAF_Gold_Cup
https://en.wikipedia.org/wiki/Copa_Am%C3%A9rica
https://en.wikipedia.org/wiki/2007_Copa_Am%C3%A9rica
https:/

In [5]:
df_pages = pd.DataFrame(data_pages)
df_images = pd.DataFrame(data_images)

df_pages.to_csv("data/pages.csv",sep=";", index=False)
df_images.to_csv("data/images.csv", sep=";", index=False)

In [6]:
df_pages.head(20)

,title,first_paragraph,link_principal,internal_links,timestamp
0,Lionel Messi,"Lionel Andrés""Leo""Messi[note 1](born 24 June 1...",https://en.wikipedia.org/wiki/Lionel_Messi,[https://en.wikipedia.org/wiki/Messi_(disambig...,2026-04-10 14:36:29.569590
1,2003–04 UEFA Champions League,The2003–04 UEFA Champions Leaguewas the 12th s...,https://en.wikipedia.org/wiki/2003%E2%80%9304_...,[https://en.wikipedia.org/wiki/Arena_AufSchalk...,2026-04-10 14:36:32.809528
2,2024 Leagues Cup,The2024 Leagues Cupwas the fourth edition of t...,https://en.wikipedia.org/wiki/2024_Leagues_Cup,[https://en.wikipedia.org/wiki/United_States_S...,2026-04-10 14:36:33.824309
3,Jonatan Maidana,Jonatan Ramón Maidana(born 29 July 1985) is an...,https://en.wikipedia.org/wiki/Jonatan_Maidana,[https://en.wikipedia.org/wiki/Club_Atl%C3%A9t...,2026-04-10 14:36:37.427073
4,1955–56 La Liga,The1955–56La Ligawas the 25th season since its...,https://en.wikipedia.org/wiki/1955%E2%80%9356_...,[https://en.wikipedia.org/wiki/La_Liga] | [htt...,2026-04-10 14:37:13.870906
5,1994 FIFA World Cup,The1994 FIFA World Cupwas the 15thFIFA World C...,https://en.wikipedia.org/wiki/1994_FIFA_World_Cup,[https://en.wikipedia.org/wiki/1994_World_Cup_...,2026-04-10 14:37:14.222995
6,Marco van Basten,"Marcel""Marco""van Basten[2](Dutch pronunciation...",https://en.wikipedia.org/wiki/Marco_van_Basten,[https://en.wikipedia.org/wiki/Dutch_name] | [...,2026-04-10 14:37:14.575649
7,2015 Copa América,The2015 Copa Américawas the 44th edition of th...,https://en.wikipedia.org/wiki/2015_Copa_Am%C3%...,[https://en.wikipedia.org/wiki/Portuguese_lang...,2026-04-10 14:37:14.915429
8,Ian Rush,Ian James Rush(born 20 October 1961) is a Wels...,https://en.wikipedia.org/wiki/Ian_Rush,[https://en.wikipedia.org/wiki/Order_of_the_Br...,2026-04-10 14:37:15.257652
9,1932–33 La Liga,The1932–33La Ligaseason began on 27 November 1...,https://en.wikipedia.org/wiki/1932%E2%80%9333_...,[https://en.wikipedia.org/wiki/La_Liga] | [htt...,2026-04-10 14:37:15.576556
